In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import zarr
import tqdm

## Model Architecture

In [2]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        
        # Initialize final layer to predict zero displacement
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

## Dataset Loader

In [3]:
class SupervisedDisplacementDataset(Dataset):
    def __init__(self, moving_path, fixed_path, dvf_gt_path):
        super().__init__()
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_arr = zarr.open(fixed_path, mode='r')
        self.dvf_gt_arr = zarr.open(dvf_gt_path, mode='r')
        
        # Check for consistent lengths
        self.num_images = self.moving_arr.shape[3]
        assert self.fixed_arr.shape[3] == self.num_images and self.dvf_gt_arr.shape[3] == self.num_images, \
            "The number of images and DVFs must match across all Zarr arrays."
            
        # Pre-calculate padding requirements
        self.original_shape = self.moving_arr.shape[:3] # H, W, D
        self.padded_shape = list(self.original_shape)
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
    
    def __len__(self):
        return self.num_images

    def _pad_tensor(self, tensor):
        # Assumes tensor is (C, D, H, W)
        pad_d = self.padded_shape[2] - tensor.shape[1]
        pad_h = self.padded_shape[0] - tensor.shape[2]
        pad_w = self.padded_shape[1] - tensor.shape[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

    def __getitem__(self, idx):
        # Load data from Zarr arrays as numpy arrays
        moving_np = self.moving_arr[..., idx]
        fixed_np = self.fixed_arr[..., idx]
        dvf_gt_np = self.dvf_gt_arr[:, :, :, idx, :] # Shape (H, W, D, 3)

        # Convert to tensors
        moving_tensor = torch.from_numpy(moving_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        fixed_tensor = torch.from_numpy(fixed_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        dvf_gt_tensor = torch.from_numpy(dvf_gt_np.astype(np.float32)).permute(3, 2, 0, 1)

        # Apply padding
        moving_padded = self._pad_tensor(moving_tensor)
        fixed_padded = self._pad_tensor(fixed_tensor)
        dvf_gt_padded = self._pad_tensor(dvf_gt_tensor)

        return moving_padded, fixed_padded, dvf_gt_padded

## Training

Data pathes

In [4]:
MOVING_PATH = "../DCE_codeset/MRI-Datasets/DCE"
FIXED_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
DVF_GT_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr"

In [5]:
MODEL_LOAD_PATH = "dvf_model_01_1e-5_100ep.pth"
MODEL_SAVE_PATH = "./dvf_model_01_1e-5_200ep.pth"
BATCH_SIZE = 8
LEARNING_RATE = 1e-5
NUM_EPOCHS = 100
VALIDATION_SPLIT = 0.2


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

full_dataset = SupervisedDisplacementDataset(MOVING_PATH, FIXED_PATH, DVF_GT_PATH)

test_size = int(VALIDATION_SPLIT * len(full_dataset))
train_size = len(full_dataset) - test_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

model = UNet3D().to(device)

if os.path.exists(MODEL_LOAD_PATH):
    model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=device))
    print(f"Loaded weights from {MODEL_LOAD_PATH}")
    
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()

scaler = torch.amp.GradScaler(device=device)


print(f"Starting supervised training on {device}...")
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    
    progress_bar = tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)
    for moving_batch, fixed_batch, dvf_gt_batch in progress_bar:
        moving_batch = moving_batch.to(device)
        fixed_batch = fixed_batch.to(device)
        dvf_gt_batch = dvf_gt_batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        # forward pass
        with torch.amp.autocast(device_type=device.type):
            predicted_dvf = model(fixed_batch, moving_batch)
            loss = loss_fn(predicted_dvf, dvf_gt_batch)
        
        # backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.6f}")
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average MSE Loss: {avg_loss:.6f}")

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Training complete. Model saved to: {MODEL_SAVE_PATH}")

Loaded weights from dvf_model_01_1e-5_100ep.pth
Starting supervised training on cuda...


Epoch 1 - Average MSE Loss: 0.005675


Epoch 2 - Average MSE Loss: 0.005642


Epoch 3 - Average MSE Loss: 0.005603


Epoch 4 - Average MSE Loss: 0.005558


...

Epoch 96 - Average MSE Loss: 0.003634


Epoch 97 - Average MSE Loss: 0.003610


Epoch 98 - Average MSE Loss: 0.003607


Epoch 99 - Average MSE Loss: 0.003593


Epoch 100 - Average MSE Loss: 0.003582
Training complete. Model saved to: ./dvf_model_01_1e-5_200ep.pth


In [6]:
test_indices = test_dataset.indices
indices_save_path = "test_indices2.pth"
torch.save(test_indices, indices_save_path)

In [ ]:
import os
import time

print("Alle Berechnungen sind abgeschlossen.")
print("Der Computer wird in 60 Sekunden heruntergefahren...")
print("Drücken Sie Strg+C in der Konsole, in der Jupyter läuft, um abzubrechen.")

# Eine kleine Wartezeit, um den Vorgang ggf. noch abbrechen zu können
time.sleep(60) 

# Der Befehl zum Herunterfahren
# 'sudo' ist nötig, aber dank der Konfiguration wird kein Passwort benötigt.
# 'now' bedeutet, dass der PC sofort heruntergefahren wird.
shutdown_command = "sudo shutdown now"

print("Sende Befehl zum Herunterfahren...")
try:
    os.system(shutdown_command)
except Exception as e:
    print(f"Fehler beim Herunterfahren: {e}")
    print("Möglicherweise müssen die sudo-Rechte wie beschrieben konfiguriert werden.")

Alle Berechnungen sind abgeschlossen.
Der Computer wird in 60 Sekunden heruntergefahren...
Drücken Sie Strg+C in der Konsole, in der Jupyter läuft, um abzubrechen.
Sende Befehl zum Herunterfahren...


sh: line 1: sudo: command not found
